# Output schemas for point data

This notebook shows how to use the `weather_tools` output-schema module when:

- **Consuming** data returned by the API/local functions (validate it conforms to the contract)
- **Integrating** with a downstream package (use the schema as a reference for expected columns)

## Sources covered

| Source | Function | Schema |
|--------|----------|--------|
| SILO PatchedPoint | `SiloAPI.get_patched_point()` | `SiloPointSchema` |
| SILO DataDrill | `SiloAPI.get_data_drill()` | `SiloPointSchema` |
| met.no forecast | `MetNoAPI.get_daily_forecast()` | `MetNoForecastSchema` |
| Merged history+forecast | `merge_historical_and_forecast()` | `MergedPointSchema` |

**Prerequisites**: set `SILO_API_KEY` to your registered SILO e-mail address.

In [1]:
import os

import pandas as pd

from weather_tools.merge_weather_data import merge_historical_and_forecast
from weather_tools.metno_api import MetNoAPI
from weather_tools.output_schemas import (
    DATE_COLUMN,
    MetNoForecastSchema,
    MergedPointSchema,
    PointMetadata,
    SiloPointSchema,
    validate_point_dataframe,
)
from weather_tools.silo_api import SiloAPI

print(f"Standard date column name: '{DATE_COLUMN}'")

Standard date column name: 'date'


## Helper: preparing a raw SILO response for schema validation

The SILO API attaches several implementation-specific columns to raw responses that are
not part of the point-data contract:

| Column pattern | Source | How to handle |
|---|---|---|
| `*_source` (e.g. `daily_rain_source`) | Data quality flags | Drop before schema validation |
| `station` | PatchedPoint location | Pass to `PointMetadata.station_code` |
| `latitude` / `longitude` | DataDrill location | Pass to `PointMetadata` |

The helper below strips these so the DataFrame matches the schema contract.

In [ ]:
def prepare_silo_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop SILO-specific extra columns that are not part of the point-data schema.

    Strips: *_source quality flags and location columns
    (station, latitude, longitude) that belong in PointMetadata instead.
    """
    drop = [
        c
        for c in df.columns
        if c.endswith("_source")
        or c in ("station", "latitude", "longitude")
    ]
    return df.drop(columns=drop, errors="ignore")

## 1. SILO PatchedPoint → `SiloPointSchema`

PatchedPoint returns station observational data with infilled gaps.

In [ ]:
STATION_CODE = "40913"  # Brisbane
VARIABLES = ["daily_rain", "max_temp", "min_temp", "vp", "radiation"]
START = "20250101"
END = "20250107"

silo = SiloAPI(log_level="DEBUG")  # or logging.DEBUG

raw_pp, pp_meta = silo.get_patched_point(
    station_code="30043",
    start_date=START,
    end_date=END,
    variables=VARIABLES,
)

print("Raw PatchedPoint columns:", raw_pp.columns.tolist())
print("\nMetadata dict:", pp_meta)
raw_pp.head()

In [19]:
raw_pp

,station,daily_rain,daily_rain_source,max_temp,max_temp_source,min_temp,min_temp_source,vp,vp_source,radiation,radiation_source,date,metadata
0,30043,0.0,25.0,43.5,25.0,24.1,25.0,18.6,25.0,25.0,42.0,2025-01-01,"{""name"": ""PROA STATION"", ""latitude"": ""-20.8944..."
1,30043,0.3,25.0,41.3,25.0,24.7,25.0,22.7,25.0,29.5,42.0,2025-01-02,NaN
2,30043,0.3,25.0,40.3,25.0,21.1,25.0,19.0,25.0,28.6,42.0,2025-01-03,NaN
3,30043,0.0,25.0,40.2,25.0,24.5,25.0,22.6,25.0,29.6,42.0,2025-01-04,NaN
4,30043,0.5,25.0,40.8,25.0,25.8,25.0,21.5,25.0,25.3,42.0,2025-01-05,NaN
5,30043,0.0,25.0,40.9,25.0,25.7,25.0,21.6,25.0,28.2,42.0,2025-01-06,NaN
6,30043,0.0,25.0,42.6,25.0,26.2,25.0,22.8,25.0,29.7,42.0,2025-01-07,NaN


In [17]:
# Prepare and validate
pp_df = prepare_silo_df(raw_pp)
print("Schema-ready columns:", pp_df.columns.tolist())

issues = validate_point_dataframe(raw_pp, SiloPointSchema)
print("\nValidation issues:", issues or "none — DataFrame is schema-conformant")

pp_df.head()

Schema-ready columns: ['daily_rain', 'max_temp', 'min_temp', 'vp', 'radiation', 'date']

Validation issues: ['DataFrame index must be RangeIndex, got Index', "Column 'metadata' found in DataFrame. Use return_metadata=True to receive metadata separately as a dict."]


,daily_rain,max_temp,min_temp,vp,radiation,date
0,0.0,43.5,24.1,18.6,25.0,2025-01-01
1,0.3,41.3,24.7,22.7,29.5,2025-01-02
2,0.3,40.3,21.1,19.0,28.6,2025-01-03
3,0.0,40.2,24.5,22.6,29.6,2025-01-04
4,0.5,40.8,25.8,21.5,25.3,2025-01-05


In [13]:
pp_meta

{'station_code': '30043',
 'date_range': {'start': '20250101', 'end': '20250107'},
 'variables': ['daily_rain', 'max_temp', 'min_temp', 'vp', 'radiation'],
 'format': 'csv',
 'dataset': 'PatchedPoint'}

In [14]:
# Pair the DataFrame with a PointMetadata companion
import datetime as dt

pp_metadata = PointMetadata(
    latitude=-27.481,  # Brisbane station lat
    longitude=153.039,
    station_code=pp_meta["station_code"],
    source="silo_patched_point",
    start_date=dt.date(2025, 1, 1),
    end_date=dt.date(2025, 1, 7),
    variables=pp_meta["variables"],
)

print(pp_metadata.model_dump())

{'latitude': -27.481, 'longitude': 153.039, 'station_code': '30043', 'source': 'silo_patched_point', 'start_date': datetime.date(2025, 1, 1), 'end_date': datetime.date(2025, 1, 7), 'variables': ['daily_rain', 'max_temp', 'min_temp', 'vp', 'radiation']}


## 2. SILO DataDrill → `SiloPointSchema`

DataDrill returns gridded interpolated data for any lat/lon coordinate.

In [ ]:
LAT, LON = -27.5, 153.0

raw_dd, dd_meta = silo.get_data_drill(
    latitude=LAT,
    longitude=LON,
    start_date=START,
    end_date=END,
    variables=VARIABLES,
)

print("Raw DataDrill columns:", raw_dd.columns.tolist())
raw_dd.head()

In [20]:
dd_df = prepare_silo_df(raw_dd)

issues = validate_point_dataframe(dd_df, SiloPointSchema)
print("Validation issues:", issues or "none — DataFrame is schema-conformant")

dd_df.head()

Validation issues: none — DataFrame is schema-conformant


,daily_rain,max_temp,min_temp,vp,radiation,date
0,0.0,30.2,20.2,21.2,18.9,2025-01-01
1,2.0,28.1,21.9,24.8,13.5,2025-01-02
2,1.7,29.6,18.9,23.8,24.2,2025-01-03
3,6.8,29.0,19.8,21.9,22.9,2025-01-04
4,1.1,29.1,17.2,21.4,26.7,2025-01-05


## 3. met.no forecast → `MetNoForecastSchema`

Met.no daily forecasts use **descriptive column names** that differ from SILO canonical
names. The schema documents the mapping.

| SILO name | Met.no name |
|---|---|
| `daily_rain` | `total_precipitation` |
| `max_temp` | `max_temperature` |
| `min_temp` | `min_temperature` |
| `vp` | `avg_relative_humidity` (converted) |
| `mslp` | `avg_pressure` |

In [21]:
metno = MetNoAPI()
forecast_df = metno.get_daily_forecast(latitude=LAT, longitude=LON, days=7)

print("Forecast columns:", forecast_df.columns.tolist())
print(f"\nDate column dtype: {forecast_df['date'].dtype}")
forecast_df

Forecast columns: ['date', 'min_temperature', 'max_temperature', 'total_precipitation', 'avg_wind_speed', 'max_wind_speed', 'avg_relative_humidity', 'avg_pressure', 'avg_cloud_fraction', 'dominant_weather_symbol']

Date column dtype: datetime64[ns]


,date,min_temperature,max_temperature,total_precipitation,avg_wind_speed,max_wind_speed,avg_relative_humidity,avg_pressure,avg_cloud_fraction,dominant_weather_symbol
0,2026-02-19,20.6,31.4,0.0,2.409091,3.8,82.695455,1014.286364,56.422727,fog
1,2026-02-20,20.1,28.5,0.0,2.575000,4.3,75.254167,1014.433333,14.387500,partlycloudy_day
2,2026-02-21,21.5,28.3,0.1,2.642105,3.8,69.157895,1012.184211,22.857895,cloudy
3,2026-02-22,20.7,27.3,1.1,2.200000,3.8,77.475000,1011.650000,29.300000,lightrainshowers_day
4,2026-02-23,21.7,27.5,1.5,2.275000,3.3,77.500000,1013.150000,53.925000,rainshowers_day
5,2026-02-24,20.8,27.1,1.6,3.475000,4.4,78.275000,1014.900000,28.500000,lightrainshowers_day
6,2026-02-25,22.8,26.9,2.1,3.250000,4.2,82.125000,1013.325000,89.650000,rain


In [22]:
# No preparation needed — get_daily_forecast already returns a clean DataFrame
issues = validate_point_dataframe(forecast_df, MetNoForecastSchema)
print("Validation issues:", issues or "none — DataFrame is schema-conformant")

Validation issues: none — DataFrame is schema-conformant


## 4. Merged history + forecast → `MergedPointSchema`

`merge_historical_and_forecast()` converts met.no columns to SILO canonical names
and adds provenance columns (`data_source`, `is_forecast`).

In [ ]:
# Pull SILO history for the week immediately before the forecast window
forecast_start = forecast_df["date"].min()
history_end = forecast_start - pd.Timedelta(days=1)
history_start = history_end - pd.Timedelta(days=6)

silo_history, _ = silo.get_data_drill(
    latitude=LAT,
    longitude=LON,
    start_date=history_start.strftime("%Y%m%d"),
    end_date=history_end.strftime("%Y%m%d"),
    variables=["daily_rain", "max_temp", "min_temp"],
)

merged_df = merge_historical_and_forecast(
    silo_data=silo_history,
    metno_data=forecast_df,
    overlap_strategy="prefer_silo",
)

print("Merged columns:", merged_df.columns.tolist())
merged_df[["date", "daily_rain", "max_temp", "min_temp", "data_source", "is_forecast"]]

In [24]:
issues = validate_point_dataframe(merged_df, MergedPointSchema)
print("Validation issues:", issues or "none — DataFrame is schema-conformant")

Validation issues: none — DataFrame is schema-conformant


## 5. Using schemas as an integration reference

`SiloPointSchema.column_descriptions()` returns a machine-readable dictionary that
downstream packages can use to discover available variables and their units.

In [25]:
descriptions = SiloPointSchema.column_descriptions()

# Formatted for reading
print(f"{'Column':<25} {'Description'}")
print("-" * 65)
for col, desc in descriptions.items():
    if desc:  # skip date which has no unit annotation
        print(f"{col:<25} {desc}")

Column                    Description
-----------------------------------------------------------------
date                      Date of observation
daily_rain                Daily rainfall (mm)
monthly_rain              Monthly rainfall (mm)
max_temp                  Maximum temperature (°C)
min_temp                  Minimum temperature (°C)
vp                        Vapour pressure (hPa)
vp_deficit                Vapour pressure deficit (hPa)
rh_tmax                   Relative humidity at time of max temperature (%)
rh_tmin                   Relative humidity at time of min temperature (%)
mslp                      Mean sea level pressure (hPa)
evap_pan                  Class A pan evaporation (mm)
evap_syn                  Synthetic estimate of open-water evaporation (mm)
evap_comb                 Combination evaporation (mm)
evap_morton_lake          Morton's shallow lake evaporation (mm)
radiation                 Solar exposure, direct and diffuse (MJ/m²)
et_short_crop           

In [26]:
# Downstream packages can also inspect required columns programmatically
print("SiloPointSchema required columns :", SiloPointSchema.REQUIRED_COLUMNS)
print("MetNoForecastSchema required columns:", MetNoForecastSchema.REQUIRED_COLUMNS)
print("MergedPointSchema required columns  :", MergedPointSchema.REQUIRED_COLUMNS)

SiloPointSchema required columns : ['date']
MetNoForecastSchema required columns: ['date']
MergedPointSchema required columns  : ['date', 'data_source', 'is_forecast']


## 6. Module constants for safe column access

Use the exported constants instead of hard-coded strings to avoid typos.

In [27]:
from weather_tools.output_schemas import (
    DATA_SOURCE_COLUMN,
    DATE_COLUMN,
    IS_FORECAST_COLUMN,
)

# Using constants when slicing DataFrames prevents silent bugs if column names change
forecast_rows = merged_df[merged_df[IS_FORECAST_COLUMN] == True]
silo_rows = merged_df[merged_df[DATA_SOURCE_COLUMN] == "silo"]

print(f"Forecast rows  ({IS_FORECAST_COLUMN}=True): {len(forecast_rows)}")
print(f"Historical rows ({DATA_SOURCE_COLUMN}='silo'): {len(silo_rows)}")
print(f"\nDate range via constant: {merged_df[DATE_COLUMN].min().date()} → {merged_df[DATE_COLUMN].max().date()}")

Forecast rows  (is_forecast=True): 7
Historical rows (data_source='silo'): 7

Date range via constant: 2026-02-12 → 2026-02-25
